# Prueba de los pesos entrenados en Colab sobre el vídeo

Usa los pesos `best.pt` del entrenamiento de YOLO11n en Colab (60 épocas, `imgsz=1280`) sobre el vídeo `C0142.MP4`.

In [ ]:
import cv2
from pathlib import Path

from ultralytics import YOLO

In [ ]:
# Pesos del entrenamiento de Colab (P4/runs/matriculas_yolo11n_1280, del zip descargado de Colab)
weights = Path("runs/matriculas_yolo11n_1280/weights/best.pt")
filename = "../videos/C0142.MP4"
IMGSZ = 1280  # resolución de inferencia (el entrenamiento fue a 1280); prueba 640, 960, 1280
CONF = 0.15   # umbral de confianza, más bajo = más detecciones lejanas (y más falsos positivos)

assert weights.exists(), f"No se encuentran los pesos: {weights.resolve()}"

model = YOLO(str(weights))
print(model.names)

Visualización con las utilidades de ultralytics (pulsa `q` o cierra la ventana para parar)

In [ ]:
results = model(filename, show=True, imgsz=IMGSZ, conf=CONF)

cv2.destroyAllWindows()

Visualización propia con OpenCV, guardando además el resultado en `videos/salida_C0142.mp4`

In [ ]:
salida = "../videos/salida_C0142.mp4"

vid = cv2.VideoCapture(filename)
fps = vid.get(cv2.CAP_PROP_FPS) or 25
w = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(salida, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

while True:
    ret, img = vid.read()
    if not ret:
        break

    results = model(img, imgsz=IMGSZ, conf=CONF, verbose=False)

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            nombre = model.names[int(box.cls[0])]

            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 255), 2)
            cv2.putText(img, f"{nombre} {conf:.2f}", (x1, max(y1 - 8, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)

    out.write(img)
    cv2.imshow("Matriculas", img)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

vid.release()
out.release()
cv2.destroyAllWindows()